# 第9章 模拟与期权定价 · 课堂代码

> 本 notebook 与课件《Python金融数据分析 · 第9章 模拟与期权定价》配套。
> 内容改编自《Python金融大数据分析（第2版）》第12章（推断统计学）。

**本章主线**：随机数 → 模拟股价（静态/动态）→ 蒙特卡洛给欧式期权定价。

全程参数：$S_0=100,\ r=5\%,\ \sigma=25\%,\ T=1,\ K=105$

In [ ]:
import math
import numpy as np
import numpy.random as npr
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False

## 1. 随机数：模拟的原材料

`numpy.random` 提供伪随机数。**先设种子**，结果才能复现。

In [ ]:
np.random.seed(1000)          # 固定种子：结果可重现
npr.rand(5)                   # 5个 [0,1) 均匀分布随机数

In [ ]:
a, b = 5, 10
npr.rand(5) * (b - a) + a     # 线性变换到 [5, 10) 区间

In [ ]:
# 标准正态随机数是金融模拟的主角
sn = npr.standard_normal(1000000)   # 一百万个
print("均值:", round(float(sn.mean()), 6), "（理论值 0）")
print("标准差:", round(float(sn.std()), 6), "（理论值 1）")

In [ ]:
# 四种常见分布的直方图
sample_size = 5000
np.random.seed(1000)
rn1 = npr.standard_normal(sample_size)          # 标准正态
rn2 = npr.normal(100, 20, sample_size)          # 正态(100, 20)
rn3 = npr.chisquare(df=0.5, size=sample_size)   # 卡方
rn4 = npr.poisson(lam=1.0, size=sample_size)    # 泊松（罕见事件）

fig, axes = plt.subplots(2, 2, figsize=(8, 5))
axes[0, 0].hist(rn1, bins=25); axes[0, 0].set_title("标准正态分布")
axes[0, 1].hist(rn2, bins=25); axes[0, 1].set_title("正态分布(100, 20)")
axes[1, 0].hist(rn3, bins=25); axes[1, 0].set_title("卡方分布(df=0.5)")
axes[1, 1].hist(rn4, bins=15); axes[1, 1].set_title("泊松分布(λ=1)")
for ax in axes.ravel():
    ax.set_ylabel("频数")
fig.tight_layout()
plt.show()

## 2. 静态模拟：一步模拟到期日股价

BSM 框架下，到期日价格为

$$S_T = S_0 \exp\!\left(\left(r-\frac{\sigma^2}{2}\right)T + \sigma\sqrt{T}\,z\right),\quad z\sim N(0,1)$$

抽 $I$ 个 $z$，就得到 $I$ 个可能的到期价格。

In [ ]:
S0, r, sigma, T = 100.0, 0.05, 0.25, 1.0   # 市场参数
I = 10000                                   # 模拟次数

np.random.seed(1000)
ST = S0 * np.exp((r - 0.5 * sigma ** 2) * T
         + sigma * np.sqrt(T) * npr.standard_normal(I))

print("模拟均值:", round(float(ST.mean()), 2), " 理论均值 S0*e^{rT} =",
      round(S0 * math.exp(r * T), 2))
print("模拟标准差:", round(float(ST.std()), 2))
print("最小值:", round(float(ST.min()), 2), " 最大值:", round(float(ST.max()), 2))

In [ ]:
# 到期价格呈对数正态分布：恒为正、右偏
plt.figure(figsize=(7, 4))
plt.hist(ST, bins=50)
plt.axvline(S0 * math.exp(r * T), color="red", linestyle="--",
            label="理论均值 $S_0e^{rT}$")
plt.xlabel("到期指数水平")
plt.ylabel("频数")
plt.title("静态模拟：到期日指数水平（对数正态分布）")
plt.legend()
plt.tight_layout()
plt.show()

## 3. 动态模拟：逐步走出完整路径

把 $T$ 切成 $M$ 步，每步 $\Delta t = T/M$，用欧拉格式逐步推进：

$$S_t = S_{t-\Delta t}\,\exp\!\left(\left(r-\frac{\sigma^2}{2}\right)\Delta t + \sigma\sqrt{\Delta t}\,z_t\right)$$

**循环只走时间（50次），路径方向整体向量化（1万条一次算完）。**

In [ ]:
M = 50                  # 时间步数
dt = T / M              # 每步长度
I = 10000               # 路径条数

np.random.seed(1000)
S = np.zeros((M + 1, I))    # (51, 10000) 大数组
S[0] = S0                   # 所有路径从 100 出发
for t in range(1, M + 1):
    S[t] = S[t - 1] * np.exp((r - 0.5 * sigma ** 2) * dt
             + sigma * np.sqrt(dt) * npr.standard_normal(I))

print("到期均值:", round(float(S[-1].mean()), 2), "（与静态模拟、理论值一致）")
print("数组形状:", S.shape)

In [ ]:
# 一万条路径里挑十条看看
plt.figure(figsize=(7, 4))
plt.plot(S[:, :10], lw=1.5)
plt.xlabel("时间步")
plt.ylabel("指数水平")
plt.title("动态模拟：几何布朗运动路径（前10条）")
plt.tight_layout()
plt.show()

## 4. 方差缩减：让随机数更"标准"

- **对偶变量**：把随机数的相反数拼进去，均值必为 0；
- **矩匹配**：直接标准化，均值 0、标准差 1。

In [ ]:
np.random.seed(2000)
sn_raw = npr.standard_normal(10000)
print("原始:   均值 %.6f  标准差 %.6f" % (sn_raw.mean(), sn_raw.std()))

np.random.seed(2000)
half = npr.standard_normal(5000)
sn_anti = np.concatenate((half, -half))      # 对偶变量
print("对偶:   均值 %.6f  标准差 %.6f" % (sn_anti.mean(), sn_anti.std()))

sn_mm = (sn_raw - sn_raw.mean()) / sn_raw.std()   # 矩匹配
print("矩匹配: 均值 %.6f  标准差 %.6f" % (sn_mm.mean(), sn_mm.std()))

## 5. 欧式期权蒙特卡洛估值

风险中性定价：$C_0 = e^{-rT}\,\mathrm{E}[h(S_T)]$，看涨期权 $h(S_T)=\max(S_T-K,0)$。

**三步曲**：① 模拟到期价 → ② 算每条路径的收益 → ③ 折现取平均。

In [ ]:
# 先看"标准答案"：BSM 解析公式
def bsm_call(S0, K, T, r, sigma):
    d1 = (math.log(S0 / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return S0 * norm.cdf(d1) - math.exp(-r * T) * K * norm.cdf(d2)

def bsm_put(S0, K, T, r, sigma):
    d1 = (math.log(S0 / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return math.exp(-r * T) * K * norm.cdf(-d2) - S0 * norm.cdf(-d1)

K = 105.0
C_ana = bsm_call(S0, K, T, r, sigma)
P_ana = bsm_put(S0, K, T, r, sigma)
print("BSM 解析价: 看涨", round(C_ana, 4), " 看跌", round(P_ana, 4))

In [ ]:
# 随机数生成器（教材风格）：返回 (M+1, I) 的随机数数组
def gen_sn(M, I):
    return npr.standard_normal((M + 1, I))

# 蒙特卡洛静态估值（只模拟到期日）
I = 50000
np.random.seed(100)
sn = gen_sn(1, I)
ST = S0 * np.exp((r - 0.5 * sigma ** 2) * T
         + sigma * np.sqrt(T) * sn[1])         # ① 模拟
hT = np.maximum(ST - K, 0)                     # ② 收益
C0 = np.exp(-r * T) * np.mean(hT)              # ③ 折现平均

print("MC 估算:", round(float(C0), 4), " 解析:", round(C_ana, 4),
      " 相对误差:", round((C0 - C_ana) / C_ana * 100, 2), "%")

In [ ]:
# 蒙特卡洛动态估值（走完整路径，可估看涨/看跌）
def gbm_mcs_dyna(K, option="call", I=50000, M=50):
    dt = T / M
    S = np.zeros((M + 1, I))
    S[0] = S0
    sn = gen_sn(M, I)
    for t in range(1, M + 1):
        S[t] = S[t - 1] * np.exp((r - 0.5 * sigma ** 2) * dt
                 + sigma * np.sqrt(dt) * sn[t])
    if option == "call":
        hT = np.maximum(S[-1] - K, 0)
    else:
        hT = np.maximum(K - S[-1], 0)
    return np.exp(-r * T) * np.mean(hT)

np.random.seed(100)
C0_dyna = gbm_mcs_dyna(K, "call")
P0_dyna = gbm_mcs_dyna(K, "put")
print("动态MC 看涨:", round(float(C0_dyna), 4), "（解析", round(C_ana, 4), "）")
print("动态MC 看跌:", round(float(P0_dyna), 4), "（解析", round(P_ana, 4), "）")

### 收敛性：路径越多，估计越准

误差按 $1/\sqrt{I}$ 缩小：精度提高 10 倍，样本要多 100 倍。

In [ ]:
# 不同路径数下的估算值与标准误差
np.random.seed(100)
for I_try in [500, 5000, 50000, 500000]:
    sn = gen_sn(1, I_try)
    ST = S0 * np.exp((r - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * sn[1])
    pv = np.exp(-r * T) * np.maximum(ST - K, 0)
    print("I=%8d: 估算值 %.4f  标准误差 %.4f" %
          (I_try, pv.mean(), pv.std() / np.sqrt(I_try)))

In [ ]:
# 收敛图：估算值逐渐贴近解析基准
Is = np.logspace(2, 5.5, 25).astype(int)
prices, ses = [], []
np.random.seed(100)
for I_try in Is:
    sn = gen_sn(1, int(I_try))
    ST = S0 * np.exp((r - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * sn[1])
    pv = np.exp(-r * T) * np.maximum(ST - K, 0)
    prices.append(pv.mean())
    ses.append(pv.std() / np.sqrt(int(I_try)))

plt.figure(figsize=(7, 4))
plt.semilogx(Is, prices, "ro-", ms=4, label="蒙特卡洛估算值")
plt.axhline(C_ana, color="blue", linestyle="--",
            label="BSM解析值 %.2f" % C_ana)
plt.fill_between(Is, np.array(prices) - np.array(ses),
                 np.array(prices) + np.array(ses),
                 color="red", alpha=0.2, label="±1个标准误差")
plt.xlabel("模拟路径数 I（对数坐标）")
plt.ylabel("期权价值")
plt.title("蒙特卡洛估值的收敛性（K=105）")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 全面体检：行权价范围对比（静态MC vs 解析）
def gbm_mcs_stat(K, I=50000):
    sn = gen_sn(1, I)
    ST = S0 * np.exp((r - 0.5 * sigma ** 2) * T
             + sigma * np.sqrt(T) * sn[1])
    return np.exp(-r * T) * np.mean(np.maximum(ST - K, 0))

k_list = np.arange(80.0, 120.1, 5.0)
stat_res, anal_res = [], []
np.random.seed(100)
for K_try in k_list:
    stat_res.append(gbm_mcs_stat(K_try))
    anal_res.append(bsm_call(S0, K_try, T, r, sigma))
stat_res = np.array(stat_res); anal_res = np.array(anal_res)

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(7, 5))
ax1.plot(k_list, anal_res, "b", label="解析值")
ax1.plot(k_list, stat_res, "ro", label="蒙特卡洛")
ax1.set_ylabel("欧式看涨期权价值")
ax1.legend()
ax1.set_ylim(bottom=0)
ax2.bar(k_list - 0.5, (stat_res - anal_res) / anal_res * 100, 1.0)
ax2.set_xlabel("行权价 K")
ax2.set_ylabel("相对误差 (%)")
ax2.set_xlim(75, 125)
fig.tight_layout()
plt.show()

print("最大相对误差: %.2f%%" % np.max(np.abs((stat_res - anal_res) / anal_res * 100)))

## 课后任务

1. 用静态模拟估计**看跌期权**价值（收益为 $\max(K-S_T,0)$，$K=105$），并与解析值 9.88 对比。
2. 把波动率从 0.25 提高到 0.40，重新定价看涨期权，观察价格变化方向并解释。
3. （选做）分别用 $I=10^3,10^4,10^5,10^6$ 条路径定价，记录标准误差，验证 $1/\sqrt{I}$ 规律。